In [3]:
pip install duckdb

   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ------------- -------------------------- 4.5/13.1 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------- 13.1/13.1 MB 52.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import duckdb
import pandas as pd

# Connect to DuckDB and load cleaned dataset
conn = duckdb.connect()
conn.execute("CREATE TABLE supply_chain AS SELECT * FROM read_csv_auto('dataco_cleaned.csv')")

print("Table created successfully!")
print(conn.execute("SELECT COUNT(*) as total_rows FROM supply_chain").fetchdf())

Table created successfully!
   total_rows
0      180519


# Top 10 markets by total sales

In [5]:
# Top 10 markets by total sales
query = """
    SELECT 
        Market,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(AVG("Days for shipping (real)"), 2) AS avg_shipping_days
    FROM supply_chain
    GROUP BY Market
    ORDER BY total_sales DESC
"""

result = conn.execute(query).fetchdf()
print(result)

         Market  total_orders  total_sales  avg_shipping_days
0        Europe         50252  10872396.80               3.49
1         LATAM         51594  10277612.84               3.51
2  Pacific Asia         41260   8273743.74               3.50
3          USCA         25799   5066528.71               3.48
4        Africa         11614   2294452.93               3.51


In [6]:
# Late delivery rate by shipping mode
query = """
    SELECT 
        "Shipping Mode",
        COUNT(*) AS total_orders,
        SUM(CASE WHEN "Late_delivery_risk" = 1 THEN 1 ELSE 0 END) AS late_orders,
        ROUND(SUM(CASE WHEN "Late_delivery_risk" = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS late_rate_pct
    FROM supply_chain
    GROUP BY "Shipping Mode"
    HAVING COUNT(*) > 1000
    ORDER BY late_rate_pct DESC
"""

result = conn.execute(query).fetchdf()
print(result)

    Shipping Mode  total_orders  late_orders  late_rate_pct
0     First Class         27814      26513.0          95.32
1    Second Class         35216      26987.0          76.63
2        Same Day          9737       4454.0          45.74
3  Standard Class        107752      41023.0          38.07


 "Premium shipping modes (First Class, Second Class) show significantly higher late delivery rates than Standard Class. First Class has a 95% late rate, likely because tighter delivery windows leave no buffer for disruptions. This suggests the company should either re-evaluate its promised delivery windows or improve capacity planning for express shipments."

In [8]:
# Orders above average sales per market
query = """
    SELECT 
        Market,
        "Order Id",
        Sales,
        ROUND((SELECT AVG(Sales) FROM supply_chain), 2) AS avg_sales_global,
        ROUND(Sales - (SELECT AVG(Sales) FROM supply_chain), 2) AS above_avg_by
    FROM supply_chain
    WHERE Sales > (SELECT AVG(Sales) FROM supply_chain)
    ORDER BY Sales DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result)

   Market  Order Id       Sales  avg_sales_global  above_avg_by
0  Europe     68736  1999.98999            203.77       1796.22
1  Europe     68848  1999.98999            203.77       1796.22
2  Europe     68806  1999.98999            203.77       1796.22
3  Europe     68766  1999.98999            203.77       1796.22
4  Europe     68703  1999.98999            203.77       1796.22
5  Europe     68883  1999.98999            203.77       1796.22
6  Europe     68859  1999.98999            203.77       1796.22
7  Europe     68778  1999.98999            203.77       1796.22
8  Europe     68722  1999.98999            203.77       1796.22
9  Europe     68724  1999.98999            203.77       1796.22


"The highest order value (~$2,000) is nearly 10x above the global average of $203. The concentration of top orders in Europe confirms it's the key market not just by volume but also by individual transaction value."

In [10]:
# Late delivery rate by region using CTE
query = """
    WITH regional_stats AS (
        SELECT 
            "Order Region",
            COUNT(*) AS total_orders,
            SUM(Late_delivery_risk) AS late_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct,
            ROUND(AVG("Days for shipping (real)"), 2) AS avg_shipping_days
        FROM supply_chain
        GROUP BY "Order Region"
    )
    SELECT *
    FROM regional_stats
    WHERE total_orders > 500
    ORDER BY late_rate_pct DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result)

     Order Region  total_orders  late_orders  late_rate_pct  avg_shipping_days
0  Central Africa          1677        972.0          57.96               3.56
1      South Asia          7731       4350.0          56.27               3.50
2     East Africa          1852       1036.0          55.94               3.51
3  Western Europe         27109      15140.0          55.85               3.50
4  South of  USA           4045       2256.0          55.77               3.49
5  Eastern Europe          3920       2182.0          55.66               3.50
6     East of USA          6915       3849.0          55.66               3.50
7  Southeast Asia          9539       5297.0          55.53               3.50
8    Central Asia           553        306.0          55.33               3.42
9       West Asia          6009       3322.0          55.28               3.49


 "Late deliveries are not a regional issue — they're a systemic problem. Since avg_shipping_days is identical across all regions (~3.5 days) but late rates exceed 55% everywhere, the root cause is likely unrealistic promised delivery windows rather than regional logistics inefficiency."

In [11]:
# Regional performance ranking using Window Functions
query = """
    WITH regional_stats AS (
        SELECT 
            "Order Region",
            "Order Country",
            COUNT(*) AS total_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct,
            ROUND(AVG(Sales), 2) AS avg_sales
        FROM supply_chain
        GROUP BY "Order Region", "Order Country"
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (PARTITION BY "Order Region" ORDER BY late_rate_pct DESC) AS rank_in_region,
            ROUND(AVG(late_rate_pct) OVER (PARTITION BY "Order Region"), 2) AS avg_late_rate_region
        FROM regional_stats
    )
    SELECT *
    FROM ranked
    WHERE rank_in_region = 1
    ORDER BY late_rate_pct DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result)

      Order Region        Order Country  total_orders  late_rate_pct  \
0      West Africa    Sáhara Occidental             2          100.0   
1       South Asia                Bután             5          100.0   
2      East Africa        Sudán del Sur             5          100.0   
3      West Africa  República de Gambia             5          100.0   
4      East Africa              Eritrea             2          100.0   
5        West Asia              Armenia             5          100.0   
6   Southeast Asia                 Laos             6          100.0   
7  Southern Africa          Suazilandia             5          100.0   
8   Western Europe           Luxemburgo            10          100.0   
9   Central Africa    Guinea Ecuatorial             2          100.0   

   avg_sales  rank_in_region  avg_late_rate_region  
0     299.98               1                 53.52  
1     153.92               1                 64.14  
2     175.99               1                 57.

"100% late rate sounds critical but with only 2-5 orders it's statistically meaningless. Small sample sizes always produce extreme percentages — this is why filtering by minimum order count is essential before drawing any conclusions."

In [13]:
# Regional performance ranking - filtered for statistical reliability
query = """
    WITH regional_stats AS (
        SELECT 
            "Order Region",
            "Order Country",
            COUNT(*) AS total_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct,
            ROUND(AVG(Sales), 2) AS avg_sales
        FROM supply_chain
        GROUP BY "Order Region", "Order Country"
        HAVING COUNT(*) > 200
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (PARTITION BY "Order Region" ORDER BY late_rate_pct DESC) AS rank_in_region,
            ROUND(AVG(late_rate_pct) OVER (PARTITION BY "Order Region"), 2) AS avg_late_rate_region
        FROM regional_stats
    )
    SELECT *
    FROM ranked
    WHERE rank_in_region = 1
    ORDER BY late_rate_pct DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

      Order Region                    Order Country  total_orders  late_rate_pct  avg_sales  rank_in_region  avg_late_rate_region
0    South America                          Ecuador           294          67.01     202.01               1                 57.00
1      East Africa                       Mozambique           244          63.93     196.97               1                 56.06
2   Southeast Asia                          Malasia           544          61.21     194.29               1                 55.49
3      West Africa                  Costa de Marfil           257          60.70     187.74               1                 54.62
4  Northern Europe                          Irlanda           491          60.69     237.64               1                 53.17
5     Eastern Asia                    Corea del Sur           583          60.21     211.38               1                 55.39
6   Central Africa                           Angola           306          59.80     188.4

"Ecuador, Mozambique and Malaysia show the worst on-time performance per region. However, the consistent avg_late_rate_region across all regions (53-57%) confirms a systemic issue — no region stands out positively, suggesting the root cause is in global planning rather than local logistics."

🇬🇧 Interview answer: "I always verify statistical reliability — extreme percentages from small sample sizes are a classic data analysis trap. That's why I filter by minimum order count before drawing conclusions."

In [14]:
# Monthly late delivery trend with rolling average using CTE + Window Functions
query = """
    WITH monthly_stats AS (
        SELECT 
            DATE_TRUNC('month', "order date (DateOrders)") AS order_month,
            COUNT(*) AS total_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct
        FROM supply_chain
        GROUP BY DATE_TRUNC('month', "order date (DateOrders)")
    )
    SELECT 
        order_month,
        total_orders,
        late_rate_pct,
        ROUND(AVG(late_rate_pct) OVER (
            ORDER BY order_month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_avg_3m
    FROM monthly_stats
    ORDER BY order_month
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

   order_month  total_orders  late_rate_pct  rolling_avg_3m
0   2015-01-01          5322          54.11           54.11
1   2015-02-01          4729          54.85           54.48
2   2015-03-01          5362          54.76           54.57
3   2015-04-01          5126          53.84           54.48
4   2015-05-01          5357          55.09           54.56
5   2015-06-01          5134          54.11           54.35
6   2015-07-01          5299          55.46           54.89
7   2015-08-01          5273          55.68           55.08
8   2015-09-01          5140          56.69           55.94
9   2015-10-01          5302          54.81           55.73
10  2015-11-01          5235          54.27           55.26
11  2015-12-01          5371          54.68           54.59
12  2016-01-01          5317          55.58           54.84
13  2016-02-01          4894          54.15           54.80
14  2016-03-01          5210          55.76           55.16
15  2016-04-01          5097          55

"I used a 3-month rolling average to smooth out monthly anomalies and reveal the underlying trend. Single-month spikes can be caused by seasonal events or one-off disruptions — the rolling average shows whether late delivery is genuinely improving or worsening over time."

 "Over the entire 2015-2017 period, the late delivery rate remains flat at 54-56% with no downward trend. The rolling average confirms this is not seasonal or one-off — it's a persistent structural problem requiring intervention at the delivery planning level, not operational fixes."

  "I used SQL to analyze 180k supply chain orders — from basic aggregations to CTEs for multi-step logic and Window Functions for regional ranking and 3-month rolling averages. Key finding: late delivery is a systemic issue affecting all regions consistently at 54-56% over 3 years, not a regional or seasonal problem."